In [2]:
import sys
print(sys.executable)

/home/julia/Desktop/seed-counter/bin/python


## Imports

In [3]:
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from sahi.slicing import slice_image
from scipy.stats import variation

## TODO
Add "Negative Samples":

    Take a few photos of the empty white paper, the ruler, and the handwritten labels with zero seeds on them.
    Add the negative sample images to the train folder.
    Create an empty .txt file for each of these images in the labels folder.

## Model Training & Image Slicing

In [4]:
# Create a new YOLO26n-OBB model from scratch
# model = YOLO("yolo26n-obb.yaml")
model = YOLO("yolo26n-obb.pt") # Using a pre-trained model

# Train the model on the dataset
train_results = model.train(
    data="data/data.yaml", 
    epochs=100,
    imgsz=768,       # The size of the "window" the CPU looks at
    batch=2,         # Low batch for CPU stability
    crop_fraction=0.2,  # FOCUS: This tells YOLO to crop a small area 
                        # instead of resizing the whole 5472px image.
                        # 0.2 means it takes a 20% window of the original size.
    mosaic=0.0,         # Turn off mosaic for very small objects (seeds)
    close_mosaic=0,     # Keeps the detail sharp
    plots=True
)

WARNING ⚠️ 'crop_fraction' is deprecated and will be removed in the future.
Ultralytics 8.4.41 🚀 Python-3.13.9 torch-2.11.0+cu130 CPU (Intel Core i7-8650U 1.90GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=768, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-obb.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=train-6, nbs=64, nms=False, opset=

### Calculation of IoU

In [9]:
def get_iou(box1, box2):
    """Calculates Intersection over Union (IoU) between two boxes [x1, y1, x2, y2]"""
    x_left = max(box1[0], box2[0])
    y_top = max(box1[1], box2[1])
    x_right = min(box1[2], box2[2])
    y_bottom = min(box1[3], box2[3])

    if x_right < x_left or y_bottom < y_top:
        return 0.0

    intersection_area = (x_right - x_left) * (y_bottom - y_top)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    return intersection_area / float(area1 + area2 - intersection_area)

### Calculation of True Pos, False Pos, False Neg

In [10]:
def calculate_tp_fp_fn(preds, gts, iou_threshold=0.5):
    """Matches predictions to ground truth to find TP, FP, and FN"""
    tp = 0
    fp = 0
    matched_gt_indices = set()

    for p_box in preds:
        best_iou = 0
        best_gt_idx = -1
        
        for i, g_box in enumerate(gts):
            if i in matched_gt_indices:
                continue
            iou = get_iou(p_box, g_box)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = i
        
        if best_iou >= iou_threshold:
            tp += 1
            matched_gt_indices.add(best_gt_idx)
        else:
            fp += 1
    
    fn = len(gts) - len(matched_gt_indices)
    return tp, fp, fn

## SAHI Inference

In [17]:
# Load the model into SAHI's wrapper
# We point to the 'best.pt' created by the training above
best_model_path = os.path.join(train_results.save_dir, 'weights/best.pt')

detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=best_model_path,
    confidence_threshold=0.5,
    device="cpu", # Change to "cuda:0" if GPU
)

# Get the list of images in your val folder
val_img_dir = "data/images/val/"
val_images = [f for f in os.listdir(val_img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]

if not val_images:
    raise FileNotFoundError("No images found in the validation folder!")

# Define the missing directory variable
export_base_dir = "sahi_results/"

# Make sure the folder actually exists so Python doesn't complain
os.makedirs(export_base_dir, exist_ok=True)

# Variables for the calculations of results. 
total_error = 0
total_gt = 0

print(f"Starting batch processing on {len(val_images)} images...")

# Loop through every image
for img_name in val_images:
    target_image_path = os.path.join(val_img_dir, img_name)
    
    # --- Load ground truth labels ---
    label_path = target_image_path.replace("images", "labels") \
                                  .replace(".jpg", ".txt") \
                                  .replace(".png", ".txt") \
                                  .replace(".jpeg", ".txt")

    gt_boxes = []
    if os.path.exists(label_path):
        # Image dimensions to denormalize the coordinates
        img_cv = cv2.imread(target_image_path)
        img_h, img_w = img_cv.shape[:2]
        with open(label_path, "r") as f:
            for line in f.readlines():
                values = list(map(float, line.strip().split()))
                # We are working with OBB labels that have 1 class + 8 coordinates
                if len(values) >= 9: 
                    coords = values[1:]  
                    xs = [c * img_w for c in coords[0::2]]
                    ys = [c * img_h for c in coords[1::2]]
                    gt_boxes.append([min(xs), min(ys), max(xs), max(ys)])

    # Run Sliced Prediction
    result = get_sliced_prediction(
        target_image_path, 
        detection_model,
        slice_height=768,
        slice_width=768,
        overlap_height_ratio=0.4, # Overlap between SAHI slices, height-wise
        overlap_width_ratio=0.4,  # Same as above but width-wise
        postprocess_type="NMS",
        postprocess_match_metric="IOU",
        postprocess_match_threshold=0.15
    )

    # Convert SAHI results to coordinate list for matching gt_boxes list
    preds = []
    for pred in result.object_prediction_list:
        b = pred.bbox
        preds.append([b.minx, b.miny, b.maxx, b.maxy])

    # Calculate True Pos (TP), False Pos (FP), False Neg (FN)
    tp, fp, fn = calculate_tp_fp_fn(preds, gt_boxes, iou_threshold=0.4) # Using 0.4 threshold for small seeds for now

    # Calculate counts
    expected_count = len(gt_boxes)
    detected_count = len(result.object_prediction_list)
    difference = detected_count - expected_count
    
    # Track totals for MAE
    total_error += abs(difference)
    total_gt += expected_count
    
    # Save visuals
    output_subdir = os.path.join(export_base_dir, os.path.splitext(img_name)[0])
    result.export_visuals(
        export_dir=output_subdir,
        hide_labels=True,         # Hide label class on image
        hide_conf=True,           # Hide confidence label on image
        rect_th=2                 # Bounding box thickness
    )

    # 5. Print the comparison
    print(f"Processed {img_name}:")
    print(f"  - Detected: {detected_count}")
    print(f"  - Expected: {expected_count}")
    print(f"  - TP: {tp} | FP: {fp} | FN: {fn}")
    
    # Avoid division by zero if an image has no labels
    pct_diff = (difference / expected_count * 100) if expected_count > 0 else 0
    print(f"  - Difference: {difference:+d} ({pct_diff:.1f}%)")

    # Check if difference is within the acceptable range (+- 10% of expected seed count)
    expected_range = 10 # 10% difference based on discussion with Maria
    range_acceptance = "WITHIN ACCEPTABLE RANGE" if abs(pct_diff) <= expected_range else "OUTSIDE OF ACCEPTABLE RANGE"
    print(range_acceptance)
    print("-" * 30)

# Final Summary Calculation
mae = total_error / len(val_images) if val_images else 0

print("\n==== FINAL RESULTS ====")
pct_total_error = (total_error / total_gt * 100) if expected_count > 0 else 0
print(f"Total difference: {total_error} ({pct_total_error:.1f}%)")
print(f"Mean Absolute Error (MAE): {mae:.2f} seeds per image")

Starting batch processing on 6 images...
Performing prediction on 96 slices.
Processed IMG_0059.jpg:
  - Detected: 154
  - Expected: 148
  - TP: 111 | FP: 43 | FN: 37
  - Difference: +6 (4.1%)
WITHIN ACCEPTABLE RANGE
------------------------------
Performing prediction on 96 slices.
Processed IMG_0082.jpg:
  - Detected: 55
  - Expected: 47
  - TP: 39 | FP: 16 | FN: 8
  - Difference: +8 (17.0%)
OUTSIDE OF ACCEPTABLE RANGE
------------------------------
Performing prediction on 96 slices.
Processed IMG_0072.jpg:
  - Detected: 220
  - Expected: 211
  - TP: 183 | FP: 37 | FN: 28
  - Difference: +9 (4.3%)
WITHIN ACCEPTABLE RANGE
------------------------------
Performing prediction on 96 slices.
Processed IMG_0044.jpg:
  - Detected: 99
  - Expected: 122
  - TP: 72 | FP: 27 | FN: 50
  - Difference: -23 (-18.9%)
OUTSIDE OF ACCEPTABLE RANGE
------------------------------
Performing prediction on 96 slices.
Processed IMG_0057.jpg:
  - Detected: 238
  - Expected: 556
  - TP: 177 | FP: 61 | FN: 37

## Uncertainty Heatmaps

In [ ]:
results = model("data/images/val/", save=True)

# color code by confidence
def get_color(score):
    if score > 0.7:
        return (0, 255, 0) # high confidence: green
    elif score > 0.4:
        return (0, 255, 255) # medium confidence: yellow
    else:
        return (0, 0, 255) # low confidence: red

for r in results:
    
    if r.obb is None:
        continue

    img = r.orig_img.copy()
    obb = r.obb
    
    for box, score, cls in zip(obb.xyxyxyxy, obb.conf, obb.cls):
        pts = np.array(box, dtype=np.int32)

        color = get_color(float(score))
        cv2.polylines(img, [pts], True, color, 2)

    # convert BGR to RGB for matplotlib
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(6,6))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

In [ ]:
# 1. Create a blank accumulation mask
heatmap_mask = np.zeros((r.orig_img.shape[0], r.orig_img.shape[1]), dtype=np.float32)

for box, score in zip(r.obb.xyxyxyxy, r.obb.conf):
    # Get the center of the OBB
    center_x = int(box[:, 0].mean())
    center_y = int(box[:, 1].mean())
    
    # Add intensity at the center point (scaled by confidence)
    heatmap_mask[center_y, center_x] += float(score)

# 2. Apply Gaussian Blur to spread the "heat"
heatmap_mask = cv2.GaussianBlur(heatmap_mask, (51, 51), 0)

# 3. Normalize and Apply Colormap
heatmap_mask = cv2.normalize(heatmap_mask, None, 0, 255, cv2.NORM_MINMAX, cv2.CV_8U)

# Alternative: highlight low-confidence detections instead of high-confidence ones
# heatmap_mask[center_y, center_x] += (1 - float(score))   

heatmap_color = cv2.applyColorMap(heatmap_mask, cv2.COLORMAP_JET)

# 4. Overlay on original image
alpha = 0.5
overlay = cv2.addWeighted(r.orig_img, 1 - alpha, heatmap_color, alpha, 0)

## Intelligent Debris Filtering

## Seed Morphology Analysis

### Extracting Seed Dimensions

In [ ]:
def extract_morphology(sahi_result, ppm=1.0):
    """
    Extracts dimensions from SAHI OBB predictions.
    ppm: Pixels Per Millimeter (calibration factor)
    """
    seeds_data = []
    
    for pred in sahi_result.object_prediction_list:
        # For OBB, the width and height represent the axes of the rotated box
        w_px = pred.bbox.w
        h_px = pred.bbox.h
        
        # Ensure 'length' is always the larger dimension
        length_px = max(w_px, h_px)
        width_px = min(w_px, h_px)
        
        # Convert to mm
        length_mm = length_px / ppm
        width_mm = width_px / ppm
        area_mm2 = length_mm * width_mm # Approximation for rectangular OBB
        
        seeds_data.append({
            'length_mm': length_mm,
            'width_mm': width_mm,
            'area_mm2': area_mm2,
            'aspect_ratio': length_mm / width_mm
        })
    
    return pd.DataFrame(seeds_data)

### Seed Variability Calculation

In [ ]:
def calculate_seed_viability(df):
    """
    Categorizes seeds based on the 30% size rule.
    """
    # Use Median to establish a more stable 'typical' seed size
    baseline_area = df['area_mm2'].median()
    threshold = baseline_area * 0.30
    
    # Categorize
    df['status'] = np.where(df['area_mm2'] <= threshold, 'Aborted', 'Active')
    
    # Summary Stats
    counts = df['status'].value_counts().to_dict()
    active_count = counts.get('Active', 0)
    aborted_count = counts.get('Aborted', 0)
    
    return active_count, aborted_count, threshold

### Morphology Batch Profile Visualization

In [ ]:
def plot_viability_report(df, threshold, active_count, aborted_count):
    plt.figure(figsize=(10, 6))
    
    # Plot Active vs Aborted with different colors
    active_seeds = df[df['status'] == 'Active']['area_mm2']
    aborted_seeds = df[df['status'] == 'Aborted']['area_mm2']
    
    plt.hist(active_seeds, bins=25, color='#2ecc71', alpha=0.7, label=f'Active ({active_count})')
    plt.hist(aborted_seeds, bins=5, color='#e74c3c', alpha=0.7, label=f'Aborted ({aborted_count})')
    
    # Add the "Cut-off" line
    plt.axvline(threshold, color='black', linestyle='--', linewidth=2)
    plt.text(threshold, plt.ylim()[1]*0.9, ' 30% Threshold', rotation=0, fontweight='bold')
    
    plt.title("Seed Viability & Morphological Profile")
    plt.xlabel("Seed Area (mm²)")
    plt.ylabel("Frequency")
    plt.legend()
    
    # Add a 'Health Index' box
    health_ratio = (active_count / (active_count + aborted_count)) * 100
    plt.figtext(0.15, 0.8, f"Batch Health: {health_ratio:.1f}%", 
                bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=1'))
    
    plt.savefig("viability_report.png")